# Test buffered built-up-area lookup on 10 random benches

This notebook samples 10 bench points from `data_extraction/data.json` and checks each against polygons in `os_open_built_up_areas.gpkg`. The lookup is local and does not require an internet connection.

Change `RANDOM_SEED` to select another sample or `BUFFER_M` to adjust the edge distance.

In [1]:
import importlib.util
import json
import random
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve paths relative to the repository, even if the notebook kernel starts elsewhere.
WORKSPACE = Path.cwd()
while WORKSPACE != WORKSPACE.parent and not (WORKSPACE / 'data_extraction' / 'data.json').exists():
    WORKSPACE = WORKSPACE.parent
DATA_DIR = WORKSPACE / 'data_extraction'
DATA_PATH = DATA_DIR / 'data.json'
SCRIPT_PATH = DATA_DIR / 'identify_settlement.py'
BUILT_UP_PATH = DATA_DIR / 'os_open_built_up_areas.gpkg'

if not DATA_PATH.exists() or not SCRIPT_PATH.exists() or not BUILT_UP_PATH.exists():
    raise FileNotFoundError('Expected data.json, identify_settlement.py, and os_open_built_up_areas.gpkg in data_extraction/.')

# Load the lookup module without invoking its command-line entrypoint.
spec = importlib.util.spec_from_file_location('identify_settlement', SCRIPT_PATH)
identify_settlement = importlib.util.module_from_spec(spec)
spec.loader.exec_module(identify_settlement)

# Read the built-up polygons once so all 10 point checks reuse the local dataset.
built_up_areas = identify_settlement.load_built_up_areas(BUILT_UP_PATH)
print(f'Loaded {len(built_up_areas):,} built-up area polygons.')

# Configure the sample size and seed; edit the seed to get another sample.
SAMPLE_SIZE = 10
RANDOM_SEED = 42
BUFFER_M = 100.0  # Include benches up to 500 m beyond mapped built-up polygons.

In [2]:
# Load valid point features and preserve useful bench labels when present.
with DATA_PATH.open('r', encoding='utf-8') as handle:
    geojson = json.load(handle)

benches = []
for index, feature in enumerate(geojson.get('features', [])):
    geometry = feature.get('geometry') or {}
    coordinates = geometry.get('coordinates', [])
    if geometry.get('type') != 'Point' or len(coordinates) < 2:
        continue
    lon, lat = float(coordinates[0]), float(coordinates[1])
    properties = feature.get('properties') or {}
    benches.append({
        'feature_id': str(feature.get('id', index + 1)),
        'label': properties.get('name') or properties.get('popupContent') or '',
        'latitude': lat,
        'longitude': lon,
    })

if len(benches) < SAMPLE_SIZE:
    raise ValueError(f'Found only {len(benches)} valid bench points; need {SAMPLE_SIZE}.')

# Sample without replacement so all selected bench locations are distinct.
rng = random.Random(RANDOM_SEED)
sampled_benches = rng.sample(benches, SAMPLE_SIZE)
print(f'Sampled {len(sampled_benches)} of {len(benches)} valid benches (seed={RANDOM_SEED}).')
display(pd.DataFrame(sampled_benches))

Sampled 10 of 39881 valid benches (seed=42).


,feature_id,label,latitude,longitude
0,7484,IN MEMORY\r\nOF\r\nDr. BERNULF HODGE,51.585155,-2.098250
1,1706,IN LOVING MEMORY OF\r\nMICHAEL RICHARD MAYS\r\...,51.352909,-2.030069
2,18460,In Loving Memory of our Parents\nROBERT WILSON...,56.064949,-3.212889
3,16403,In loving memory of Laura Smallwood\n08/01/85 ...,50.544880,-4.937181
4,14955,Stephen Dean 1974 - 2018\r\nHappy times; bea...,54.433205,-0.534258
5,9365,PHILIP AND NESSIE HOPKINS\r\nREMEMBERED WITH LOVE,50.605419,-2.094320
6,6896,HIROSHIMA ・August 1945 ・NAGASAKI\r\nThese 3 Gi...,51.429821,-0.050367
7,37005,GORDON LOACH M.B.E.\r\n1929 - 2017\r\nLOVE ALWAYS,52.815559,-2.126330
8,5853,In Loving Memory Of\r\nHARRY KINAHAN\r\n'Great...,50.733334,-1.161111
9,40052,PROFESSOR MARY COTTON 1921 - 2012\r\nONE OF MA...,51.521229,-0.150790


In [3]:
# Classify each sampled bench against the local built-up-area polygons.
results = []
for index, bench in enumerate(sampled_benches, start=1):
    print(f"Checking bench {index}/{SAMPLE_SIZE}: {bench['feature_id']}")
    area_match = identify_settlement.find_built_up_area(
        bench['latitude'], bench['longitude'], built_up_areas, buffer_m=BUFFER_M
    )
    results.append({
        **bench,
        'settlement_name': area_match['name'],
        'area_hectares': area_match['areahectares'],
        'gsscode': area_match['gsscode'],
        'area_match': area_match['match'],
        'distance_to_area_m': area_match.get('distance_to_area_m'),
    })

# Display all results and optionally save them for later review.
results_df = pd.DataFrame(results)
display(results_df)
# Uncomment to save the current sample results:
# results_df.to_csv(DATA_DIR / 'built_up_area_test_sample.csv', index=False)


Looking up bench 1/10: 7484


Looking up bench 2/10: 1706
Looking up bench 3/10: 18460
Looking up bench 4/10: 16403
Looking up bench 5/10: 14955
Looking up bench 6/10: 9365
Looking up bench 7/10: 6896
Looking up bench 8/10: 37005


Looking up bench 9/10: 5853
Lookup failed for bench 5853: HTTP Error 502: Bad Gateway
Looking up bench 10/10: 40052


,feature_id,label,latitude,longitude,settlement_name,settlement_type,match_method,area_m2
0,7484,IN MEMORY\r\nOF\r\nDr. BERNULF HODGE,51.585155,-2.098250,no settlement,NaN,no_containing_settlement,None
1,1706,IN LOVING MEMORY OF\r\nMICHAEL RICHARD MAYS\r\...,51.352909,-2.030069,no settlement,NaN,no_containing_settlement,None
2,18460,In Loving Memory of our Parents\nROBERT WILSON...,56.064949,-3.212889,Burntisland,town,containing_boundary,None
3,16403,In loving memory of Laura Smallwood\n08/01/85 ...,50.544880,-4.937181,no settlement,NaN,no_containing_settlement,None
4,14955,Stephen Dean 1974 - 2018\r\nHappy times; bea...,54.433205,-0.534258,no settlement,NaN,no_containing_settlement,None
5,9365,PHILIP AND NESSIE HOPKINS\r\nREMEMBERED WITH LOVE,50.605419,-2.094320,no settlement,NaN,no_containing_settlement,None
6,6896,HIROSHIMA ・August 1945 ・NAGASAKI\r\nThese 3 Gi...,51.429821,-0.050367,no settlement,NaN,no_containing_settlement,None
7,37005,GORDON LOACH M.B.E.\r\n1929 - 2017\r\nLOVE ALWAYS,52.815559,-2.126330,no settlement,NaN,no_containing_settlement,None
8,5853,In Loving Memory Of\r\nHARRY KINAHAN\r\n'Great...,50.733334,-1.161111,NaN,NaN,lookup_error: HTTPError,None
9,40052,PROFESSOR MARY COTTON 1921 - 2012\r\nONE OF MA...,51.521229,-0.150790,no settlement,NaN,no_containing_settlement,None


A result of **no settlement** means the coordinate falls outside every built-up polygon and its configured buffer. `inside_built_up_area` marks points in a polygon; `within_buffer` marks points within 100 m outside one.